# Llama 3.2 1B Instruct on Google Colab

This notebook runs `meta-llama/Llama-3.2-1B-Instruct` on Colab with Hugging Face Transformers.

Why not use the SqueezeLLM runtime directly here:
- This repo does not ship a SqueezeLLM checkpoint for Llama 3.2 1B Instruct.
- The repo pins an older `transformers` version that predates Llama 3.2 support.
- For a first working Colab run, plain Transformers is the shortest path.

Before running:
- In Hugging Face, request or accept access for `meta-llama/Llama-3.2-1B-Instruct`.
- In Colab, set runtime to `GPU`.


In [ ]:
!nvidia-smi

!pip -q install -U transformers accelerate sentencepiece huggingface_hub datasets

%cd /content
!git clone https://github.com/SqueezeAILab/SqueezeLLM.git
%cd /content/SqueezeLLM
!git rev-parse --short HEAD


## Optional: SqueezeLLM repo setup

This section prepares the SqueezeLLM repo and CUDA extension only.

There is no working SqueezeLLM quantized run cell for `meta-llama/Llama-3.2-1B-Instruct` in this notebook because:
- the repo does not provide a packed SqueezeLLM checkpoint for this model
- the runtime in this repo was published for older model families
- `llama.py --load ...` expects a SqueezeLLM checkpoint, not the base Hugging Face weights


In [ ]:
%cd /content/SqueezeLLM
!pip -q install -e .
%cd /content/SqueezeLLM/squeezellm
!python setup_cuda.py install
%cd /content/SqueezeLLM


In [ ]:
print('SqueezeLLM runtime prepared.')
print('No official SqueezeLLM quantized checkpoint for meta-llama/Llama-3.2-1B-Instruct is included in this repo.')
print('Use the Transformers cells below for the actual model run.')


## Hugging Face login

Preferred setup:
- In Colab, open the Secrets panel and create a secret named `HF_TOKEN`.
- Put your Hugging Face read token there once.

The next cell will try these sources in order:
- `google.colab.userdata.get('HF_TOKEN')`
- environment variable `HF_TOKEN`
- manual `login()` prompt as fallback


In [ ]:
import os
from huggingface_hub import login

hf_token = None

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    pass

if not hf_token:
    hf_token = os.environ.get('HF_TOKEN')

if hf_token:
    login(token=hf_token)
    print('Logged in using stored HF_TOKEN')
else:
    print('HF_TOKEN not found in Colab Secrets or environment; falling back to manual login prompt')
    login()

In [ ]:
import torch
import transformers

print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
print('dtype:', dtype)


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = 'meta-llama/Llama-3.2-1B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=dtype,
    device_map='auto'
)

model.eval()
print('model loaded')


In [ ]:
messages = [
    {
        'role': 'system',
        'content': 'You are a concise technical assistant.'
    },
    {
        'role': 'user',
        'content': 'Explain what weight-only quantization is in 5 short bullet points.'
    },
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors='pt'
)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

prompt_len = inputs['input_ids'].shape[-1]
response = tokenizer.decode(outputs[0][prompt_len:], skip_special_tokens=True)
print(response)


In [ ]:
def chat(prompt, system='You are a concise and accurate assistant.', max_new_tokens=256):
    messages = [
        {'role': 'system', 'content': system},
        {'role': 'user', 'content': prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors='pt'
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    prompt_len = inputs['input_ids'].shape[-1]
    return tokenizer.decode(outputs[0][prompt_len:], skip_special_tokens=True)


print(chat('Write a short Python function that computes Fibonacci with iteration.'))


## Notes

- This notebook runs the base HF model, not a SqueezeLLM-quantized checkpoint.
- If you want to integrate Llama 3.2 into the repo scripts such as `llama.py`, you will need code changes and a newer `transformers` stack.
- The model supports long context, but this notebook keeps generation simple and does not benchmark full 128K context.
